# PW00 · Paper Eval Family Manifest

用途：在 Colab 冷启动环境中生成 family manifest、formal 主流程所需的双 role source event grid，以及 optional diagnostic cohort planner_conditioned_control_negative 对应的 source shard plan 与 frozen source split。

边界：仅执行 PW00，为 PW01 与 PW02 冻结真源，不执行 PW03/PW04/PW05。

## 1) 定义路径与参数

本单元只做常量定义与最小工具函数准备，不执行任何外部副作用。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW00_Paper_Eval_Family_Manifest"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW00_Paper_Eval_Family_Manifest.py"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
HF_HOME = REPO_ROOT / "huggingface_cache"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"

FAMILY_ID = "paper_eval_family_pilot_v1"
PW_BASE_CONFIG_PATH = "paper_workflow/configs/pw_base_pilot.yaml"
PROMPT_FILE = "prompts/paper_pilot_10.txt"
SEED_LIST = [100, 101, 102, 103, 104, 105, 106, 107]
SOURCE_SHARD_COUNT = 4
ATTACK_SHARD_COUNT = 16  # pilot 默认冻结为 16；None 表示回退到 SOURCE_SHARD_COUNT；显式正整数表示独立冻结 attack shard 数
RESOLVED_ATTACK_SHARD_COUNT = SOURCE_SHARD_COUNT if ATTACK_SHARD_COUNT is None else ATTACK_SHARD_COUNT

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

## 2) 挂载 Drive 并同步仓库

本单元保证冷启动时自动挂载 Drive，并把仓库同步到固定路径。

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
        "config_path": str(CONFIG_PATH),
    },
)

## 3) 安装依赖并执行 attestation bootstrap

本单元对齐主 notebook 的冷启动准备流程：安装基础依赖并确保 attestation env 可用。

In [ ]:
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)

if Path("/usr/bin/apt-get").exists():
    subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
        check=False,
        capture_output=True,
    )

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)
if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

from scripts.notebook_runtime_common import ensure_attestation_env_bootstrap, load_yaml_mapping

CFG_OBJ = load_yaml_mapping(CONFIG_PATH)
ATTESTATION_BOOTSTRAP = ensure_attestation_env_bootstrap(
    CFG_OBJ,
    DRIVE_PROJECT_ROOT,
    allow_generate=True,
    allow_missing=False,
)
print_json("attestation_env_bootstrap", ATTESTATION_BOOTSTRAP)

## 4) 运行前 precheck

本单元执行必要门禁：Drive 挂载状态、仓库同步结果、依赖路径、关键输入和参数合法性。

In [ ]:
PROMPT_PATH = Path(PROMPT_FILE).expanduser()
if not PROMPT_PATH.is_absolute():
    PROMPT_PATH = (REPO_ROOT / PROMPT_PATH).resolve()
else:
    PROMPT_PATH = PROMPT_PATH.resolve()

PW_BASE_CONFIG_RESOLVED_PATH = Path(PW_BASE_CONFIG_PATH).expanduser()
if not PW_BASE_CONFIG_RESOLVED_PATH.is_absolute():
    PW_BASE_CONFIG_RESOLVED_PATH = (REPO_ROOT / PW_BASE_CONFIG_RESOLVED_PATH).resolve()
else:
    PW_BASE_CONFIG_RESOLVED_PATH = PW_BASE_CONFIG_RESOLVED_PATH.resolve()

PROMPT_LINES = []
if PROMPT_PATH.exists():
    PROMPT_LINES = [
        line.strip()
        for line in PROMPT_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("Drive 项目根目录存在", DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW00 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("default 配置存在", CONFIG_PATH.exists(), str(CONFIG_PATH))
record_precheck("prompt 文件存在", PROMPT_PATH.exists(), str(PROMPT_PATH))
record_precheck("PW base config 存在", PW_BASE_CONFIG_RESOLVED_PATH.exists(), str(PW_BASE_CONFIG_RESOLVED_PATH))

seed_list_ok = (
    isinstance(SEED_LIST, list)
    and len(SEED_LIST) > 0
    and all(isinstance(item, int) and not isinstance(item, bool) for item in SEED_LIST)
)
attack_shard_count_ok = (
    ATTACK_SHARD_COUNT is None
    or (
        isinstance(ATTACK_SHARD_COUNT, int)
        and not isinstance(ATTACK_SHARD_COUNT, bool)
        and ATTACK_SHARD_COUNT > 0
    )
)
record_precheck("seed_list 合法", seed_list_ok, json.dumps(SEED_LIST, ensure_ascii=False))
record_precheck(
    "source_shard_count 合法",
    isinstance(SOURCE_SHARD_COUNT, int) and SOURCE_SHARD_COUNT > 0,
    str(SOURCE_SHARD_COUNT),
)
record_precheck(
    "attack_shard_count 合法（None 表示回退到 source_shard_count）",
    attack_shard_count_ok,
    str(ATTACK_SHARD_COUNT),
)
record_precheck(
    "resolved_attack_shard_count 已确定",
    attack_shard_count_ok and isinstance(RESOLVED_ATTACK_SHARD_COUNT, int) and RESOLVED_ATTACK_SHARD_COUNT > 0,
    json.dumps(
        {
            "attack_shard_count": ATTACK_SHARD_COUNT,
            "resolved_attack_shard_count": str(RESOLVED_ATTACK_SHARD_COUNT),
        },
        ensure_ascii=False,
    ),
)
record_precheck(
    "pilot prompt 数量固定为 10",
    len(PROMPT_LINES) == 10,
    json.dumps(
        {
            "prompt_count": len(PROMPT_LINES),
            "prompt_path": str(PROMPT_PATH),
        },
        ensure_ascii=False,
        sort_keys=True,
    ),
)
record_precheck(
    "pilot seed_list 固定为 [100, 101, 102, 103, 104, 105, 106, 107]",
    SEED_LIST == [100, 101, 102, 103, 104, 105, 106, 107],
    json.dumps(SEED_LIST, ensure_ascii=False),
)
record_precheck(
    "pilot source_shard_count 固定为 4",
    SOURCE_SHARD_COUNT == 4,
    str(SOURCE_SHARD_COUNT),
)
record_precheck(
    "pilot attack_shard_count 固定为 16",
    ATTACK_SHARD_COUNT == 16 and RESOLVED_ATTACK_SHARD_COUNT == 16,
    json.dumps(
        {
            "attack_shard_count": ATTACK_SHARD_COUNT,
            "resolved_attack_shard_count": RESOLVED_ATTACK_SHARD_COUNT,
        },
        ensure_ascii=False,
        sort_keys=True,
    ),
)
record_precheck(
    "attestation bootstrap 状态可用",
    str(ATTESTATION_BOOTSTRAP.get("status", "")) in {"generated", "reused", "disabled"},
    json.dumps(ATTESTATION_BOOTSTRAP, ensure_ascii=False, sort_keys=True),
)

print_json("pw00_precheck", {"items": PRECHECK_RESULTS})

hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW00 precheck failed: {hard_fail}")

## 5) 执行 PW00 CLI

本单元调用 paper_workflow/scripts/PW00_Paper_Eval_Family_Manifest.py，并解析标准 JSON summary。

In [ ]:
from scripts.notebook_runtime_common import build_repo_import_subprocess_env

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
PW00_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw00_summary.json"

COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
    "--pw-base-config-path",
    PW_BASE_CONFIG_PATH,
    "--prompt-file",
    PROMPT_FILE,
    "--seed-list",
    json.dumps(SEED_LIST),
    "--source-shard-count",
    str(SOURCE_SHARD_COUNT),
]
if ATTACK_SHARD_COUNT is not None:
    COMMAND.extend([
        "--attack-shard-count",
        str(ATTACK_SHARD_COUNT),
    ])

PW00_RESULT = subprocess.run(
    COMMAND,
    cwd=REPO_ROOT,
    env=build_repo_import_subprocess_env(repo_root=REPO_ROOT),
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if PW00_RESULT.returncode != 0:
    raise RuntimeError(
        "PW00 failed: "
        f"return_code={PW00_RESULT.returncode} stdout={PW00_RESULT.stdout} stderr={PW00_RESULT.stderr}"
    )

if not PW00_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "PW00 summary file missing after successful subprocess execution: "
        f"summary_path={PW00_SUMMARY_PATH} stdout={PW00_RESULT.stdout} stderr={PW00_RESULT.stderr}"
    )

PW00_SUMMARY = json.loads(PW00_SUMMARY_PATH.read_text(encoding="utf-8"))
print_json("pw00_summary", PW00_SUMMARY)

## 6) 回读关键产物

本单元验证 summary、manifest、event grid、shard plan 四个核心文件都已落盘。

In [ ]:
SUMMARY_PATH = Path(PW00_SUMMARY["summary_path"])
MANIFEST_PATH = Path(PW00_SUMMARY["paper_eval_family_manifest_path"])
EVENT_GRID_PATH = Path(PW00_SUMMARY["source_event_grid_path"])
SHARD_PLAN_PATH = Path(PW00_SUMMARY["source_shard_plan_path"])
ATTACK_SHARD_PLAN_PATH = Path(PW00_SUMMARY["attack_shard_plan_path"])
SPLIT_PLAN_PATH = Path(PW00_SUMMARY["source_split_plan_path"])

output_check = {
    "summary_path": {"path": str(SUMMARY_PATH), "exists": SUMMARY_PATH.exists()},
    "family_manifest": {"path": str(MANIFEST_PATH), "exists": MANIFEST_PATH.exists()},
    "source_event_grid": {"path": str(EVENT_GRID_PATH), "exists": EVENT_GRID_PATH.exists()},
    "source_shard_plan": {"path": str(SHARD_PLAN_PATH), "exists": SHARD_PLAN_PATH.exists()},
    "attack_shard_plan": {"path": str(ATTACK_SHARD_PLAN_PATH), "exists": ATTACK_SHARD_PLAN_PATH.exists()},
    "source_split_plan": {"path": str(SPLIT_PLAN_PATH), "exists": SPLIT_PLAN_PATH.exists()},
}
print_json("pw00_output_check", output_check)

missing_outputs = [name for name, info in output_check.items() if not info["exists"]]
if missing_outputs:
    raise FileNotFoundError(f"PW00 outputs missing: {missing_outputs}")

print(SUMMARY_PATH.read_text(encoding="utf-8"))

## 7) 如何扩展 source / attack 并行（显式说明）

扩展规则：
1. SOURCE_SHARD_COUNT 控制 source shard；它决定 PW01 需要消费的 source_shards/<role>/shard_xxxx 数量。
2. ATTACK_SHARD_COUNT 控制 attack shard；它决定后续 PW03 使用的 attack_shards/shard_xxxx 冻结数量。
3. 若未显式提供 ATTACK_SHARD_COUNT，则默认回退到 SOURCE_SHARD_COUNT，以保持旧行为兼容。
4. PW00 会同时冻结 positive_source 与 clean_negative 两种 source role 的 event grid、source shard plan 与 source split，并冻结 PW03 使用的 attack shard plan。
5. 再在多个 Colab 会话分别运行 PW01，并给每个会话分配 sample_role 与 shard_index。
6. 每个 source shard 写入独立目录 source_shards/<role>/shard_xxxx；后续 PW03 仍要求传入的 ATTACK_SHARD_COUNT 与 family 冻结值一致。
7. 若 shard 已完成且需要重跑，PW01 使用 --force-rerun。

In [ ]:
print("[后续并行运行说明]")
print(f"\n当前 FAMILY_ID = {FAMILY_ID}")
print(f"\nPW01 需要设置：SHARD_COUNT = {SOURCE_SHARD_COUNT}")
print(f"PW01 需要完成：SHARD_INDEX = 0-{SOURCE_SHARD_COUNT - 1}")
print("\nPW02 运行条件：上述 PW01 formal shards 全部完成后，运行 PW02 一次。")
print(f"\nPW03 需要设置：ATTACK_SHARD_COUNT = {RESOLVED_ATTACK_SHARD_COUNT}")
print(f"PW03 需要完成：ATTACK_SHARD_INDEX = 0-{RESOLVED_ATTACK_SHARD_COUNT - 1}")
print("\nPW04 运行顺序：prepare 一次 → quality_shard 全部完成 → finalize 一次。")
print("\nPW05 运行条件：PW04 finalize 完成后，运行 PW05 一次。")